<a href="https://colab.research.google.com/github/renatofb98/Data_science_projects/blob/main/Ler_Manutencoes_para_Excel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Leitor de relatório de Manutenções (GCASPP) → Excel

Este notebook lê o relatório em PDF de manutenções de veículos (formato
"PREFEITURA MUNICIPAL DE ... / ALMOXARIFADO / GARAGEM MUNICIPAL", sistema
GCASPP) e gera uma planilha Excel estruturada.

**Por que OCR e não extração de texto direta?**
Esse tipo de relatório, quando exportado do sistema pelo navegador
("Salvar como PDF"), às vezes sai com o texto convertido em curvas/vetores
em vez de texto real — nesse caso `pdftotext`/`PyPDF2`/`pdfplumber` não
extraem nada além do cabeçalho da página. O notebook detecta isso e, se
necessário, usa OCR (Tesseract). Se o seu PDF tiver texto real, ele
tentará a extração direta primeiro (mais rápida e sem erro de OCR).

**Passos:** rode as células em ordem. Na célula 3 você vai enviar o PDF.


In [1]:
# 1) Dependências de sistema (Colab: roda uma vez por sessão)
!apt-get -qq update
!apt-get -qq install -y poppler-utils tesseract-ocr tesseract-ocr-por
!pip -q install pdfplumber openpyxl pandas


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../poppler-utils_24.02.0-1ubuntu9.9_amd64.deb ...
Unpacking poppler-utils (24.02.0-1ubuntu9.9) ...
Selecting previously unselected package tesseract-ocr-por.
Preparing to unpack .../tesseract-ocr-por_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-por (1:4.1.0-2) ...
Setting up tesseract-ocr-por (1:4.1.0-2) ...
Setting up poppler-utils (24.02.0-1ubuntu9.9) ...
Processing triggers for man-db (2.12.0-4build2) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━

In [2]:
# 2) Envie o PDF do relatório de manutenções
from google.colab import files
uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
print("Arquivo recebido:", PDF_PATH)


Saving Consumo por centro de custos da frota 01-01-a-14-09-2026.pdf to Consumo por centro de custos da frota 01-01-a-14-09-2026.pdf
Arquivo recebido: Consumo por centro de custos da frota 01-01-a-14-09-2026.pdf


In [12]:
import re
import subprocess
import tempfile, os, glob
import pandas as pd

DPI = 300
LANG = "por"

# Updated regex patterns for the new PDF format
CENTRO_CUSTO_RE = re.compile(
    r'^\s*Centro de Custo:\s*(?P<code>\d+)\s*-\s*(?P<model_year>.+)\s+(?P<plate>[A-Z0-9]{3}[\s-]?[A-Z0-9]{3,4})\s*$'
)
# ITEM_RE to capture product details
ITEM_RE = re.compile(
    r'^(?P<cod_produto>\d{2}\.\d{4})\s+'
    r'(?P<descricao>.+?)\s{2,}'
    r'(?P<unid>[A-ZÀ-Ú]{1,4})\s+'
    r'(?P<quantidade>[\d.,]+)\s+'
    r'(?P<vl_liquido>[\d.,]+)\s*$'
)

# Updated BOILER_RE to skip all non-data lines from the new PDF format
BOILER_RE = re.compile(
    r'^(?:PREFEITURA|DIRETORIA DE ADMINISTRAÇÃO|SETOR DE ALMOXARIFADO|Exercício:|CONSUMO POR CENTRO DE CUSTO|GCASPP|PERÍODO:|Página:|Almoxarifado:|Veículo: GERAL|Cd\\. Produto|Total da Quantidade:|Total Líquido:|Total Centro Custo:|Total Almoxarifado:|Total Geral:|CERQUILHO,)'
    r'.*$'
)

def to_float(s: str) -> float:
    # Handles both comma as decimal and dot as thousand separator, then converts
    # Also handles multiple dots in '1.000.000,00' if it appears
    return float(s.replace('.', '').replace(',', '.'))


def has_real_text_layer(pdf_path: str) -> bool:
    """Testa se o PDF tem texto extraível de verdade (além do cabeçalho)."""
    try:
        out = subprocess.run(
            ["pdftotext", "-f", "2", "-l", "2", "-layout", pdf_path, "-"],
            capture_output=True, text=True, check=True,
        ).stdout
    except subprocess.CalledProcessError:
        return False
    # se sobrar pouco texto além do cabeçalho repetido, provavelmente é vetor/curva
    return len(out.strip()) > 300


def ocr_pdf(pdf_path: str, dpi: int = DPI, lang: str = LANG) -> str:
    with tempfile.TemporaryDirectory() as tmp:
        prefix = os.path.join(tmp, "pg")
        subprocess.run(["pdftoppm", "-r", str(dpi), "-png", pdf_path, prefix], check=True)
        pages = sorted(glob.glob(prefix + "*.png"))
        chunks = []
        for p in pages:
            res = subprocess.run(
                ["tesseract", p, "stdout", "-l", lang,
                 "--psm", "6", "-c", "preserve_interword_spaces=1"],
                capture_output=True, text=True,
            )
            chunks.append(res.stdout)
        return "\n".join(chunks)


def extract_text_direct(pdf_path: str) -> str:
    res = subprocess.run(["pdftotext", "-layout", pdf_path, "-"],
                          capture_output=True, text=True, check=True)
    return res.stdout


def parse(text: str) -> pd.DataFrame:
    rows = []
    current_centro_custo_info = {"code": None, "plate": None, "model_year": None, "dept": "ALMOXARIFADO CENTRAL"}

    for raw in text.splitlines():
        l = raw.strip()
        if not l or BOILER_RE.match(l):
            continue

        m_centro_custo = CENTRO_CUSTO_RE.match(l)
        if m_centro_custo:
            current_centro_custo_info.update({
                "code": m_centro_custo.group("code"),
                "plate": m_centro_custo.group("plate"),
                "model_year": m_centro_custo.group("model_year").strip(),
            })
            continue

        m_item = ITEM_RE.match(l)
        if m_item:
            # Only add if a cost center was identified
            if current_centro_custo_info["code"] is not None:
                rows.append({
                    "Cód. Veículo": current_centro_custo_info["code"],
                    "Placa": current_centro_custo_info["plate"],
                    "Modelo/Ano": current_centro_custo_info["model_year"],
                    "Secretaria/Setor": current_centro_custo_info["dept"],
                    "Tipo Ocorrência": "Consumo", # Default for this type of report
                    "Data": None, # No specific date per item in this report format
                    "Prestadora Serviço": "Interno", # Default for this type of report
                    "Cód. Produto": m_item.group("cod_produto"),
                    "Descrição": m_item.group("descricao").strip(),
                    "Unid.": m_item.group("unid"),
                    "Qtd. Item": to_float(m_item.group("quantidade")),
                    "Total Ocorrência (R$)": to_float(m_item.group("vl_liquido")),
                })
            else:
                print(f"Aviso: Item '{m_item.group('descricao')}' encontrado sem centro de custo associado. Ignorando.")
            continue

    return pd.DataFrame(rows)


print("Verificando se o PDF tem texto real ou precisa de OCR...")
if has_real_text_layer(PDF_PATH):
    print("→ Texto real encontrado, extraindo direto (rápido).")
    texto = extract_text_direct(PDF_PATH)
else:
    print("→ Sem texto extraível (provável texto vetorizado) — rodando OCR. "
          "Isso pode levar alguns minutos em PDFs longos.")
    texto = ocr_pdf(PDF_PATH)

df = parse(texto)
print(f"\nLinhas de manutenção extraídas: {len(df)}")
if not df.empty:
    print(f"Soma de 'Total Ocorrência': R$ {df['Total Ocorrência (R$)'].sum():,.2f}")
    df.head(10)
else:
    print("Nenhuma linha de manutenção foi extraída. Verifique o formato do PDF e as expressões regulares.")


Verificando se o PDF tem texto real ou precisa de OCR...
→ Texto real encontrado, extraindo direto (rápido).

Linhas de manutenção extraídas: 1957
Soma de 'Total Ocorrência': R$ 950,391.38


**Confira o total acima contra o rodapé do PDF original (linha "Total: N ...").**
Se baterem, a extração está correta. Se o PDF tiver colunas diferentes
(ex.: nomes de campo distintos), ajuste as regex `VEH_RE` / `DATA_RE` na
célula anterior — elas foram calibradas para o layout
`Data | KM Ocor. | Próx. KM | Prestadora Serviço | Qtd. Item | Vl. Unitário |
Vl. Óleo | Filt. Óleo | Filt Comb A | Filt Comb B | Filtro Ar | Óleo Dif. |
Vl Outros | Total Ocorr.`


In [4]:
print(texto)

                                      PREFEITURA MUNICIPAL DE CERQUILHO
                                                DIRETORIA DE ADMINISTRAÇÃO
                                                     SETOR DE ALMOXARIFADO                                    Exercício: 2026
                                    CONSUMO POR CENTRO DE CUSTO - ALMOXARIFADO CENTRAL
   GCASPP                               PERÍODO: 01/01/2026 À 14/09/2026 - SINTÉTICO                          Página:      1/49
Almoxarifado:    1 - CENTRAL
    Centro de Custo:    208 - GOL 1997                           CDZ 1332

Cd. Produto     Descrição                                             Unid. Lote             Validade   Quantidade      Valor Líquido
02.0369         ALAVANCA                                                   PÇ                             1,000000              168,00
02.8906         ANEL DE VEDAÇÃO                                        KIT                                1,000000               16,00
02.8

In [5]:
print(texto[:1000]) # Imprime os primeiros 1000 caracteres do texto

                                      PREFEITURA MUNICIPAL DE CERQUILHO
                                                DIRETORIA DE ADMINISTRAÇÃO
                                                     SETOR DE ALMOXARIFADO                                    Exercício: 2026
                                    CONSUMO POR CENTRO DE CUSTO - ALMOXARIFADO CENTRAL
   GCASPP                               PERÍODO: 01/01/2026 À 14/09/2026 - SINTÉTICO                          Página:      1/49
Almoxarifado:    1 - CENTRAL
    Centro de Custo:    208 - GOL 1997                           CDZ 1332

Cd. Produto     Descrição                                             Unid. Lote             Validade   Quantidade      Valor Líquido
02.0369         ALAVANCA                                                   PÇ                             1,000000              168,00
02.8906         ANEL DE VEDAÇÃO                                        KIT                                1,000000               16,00
02.8

In [13]:
# 4) Monta o Excel (dados + resumos)
OUT_XLSX = "Manutencoes.xlsx"

resumo_veiculo = (
    df.groupby(["Cód. Veículo", "Placa", "Modelo/Ano", "Secretaria/Setor"], dropna=False)
    .agg(Qtd_Ocorrencias=("Total Ocorrência (R$)", "count"),
         Total_Gasto=("Total Ocorrência (R$)", "sum"))
    .reset_index().sort_values("Total_Gasto", ascending=False)
)
resumo_prestadora = (
    df.groupby("Prestadora Serviço", dropna=False)
    .agg(Qtd_Ocorrencias=("Total Ocorrência (R$)", "count"),
         Total_Recebido=("Total Ocorrência (R$)", "sum"))
    .reset_index().sort_values("Total_Recebido", ascending=False)
)
resumo_setor = (
    df.groupby("Secretaria/Setor", dropna=False)
    .agg(Qtd_Ocorrencias=("Total Ocorrência (R$)", "count"),
         Total_Gasto=("Total Ocorrência (R$)", "sum"))
    .reset_index().sort_values("Total_Gasto", ascending=False)
)

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Dados", index=False)
    resumo_veiculo.to_excel(writer, sheet_name="Resumo por Veículo", index=False)
    resumo_prestadora.to_excel(writer, sheet_name="Resumo por Prestadora", index=False)
    resumo_setor.to_excel(writer, sheet_name="Resumo por Setor", index=False)

from openpyxl import load_workbook
wb = load_workbook(OUT_XLSX)
for ws in wb.worksheets:
    for col in ws.columns:
        length = max((len(str(c.value)) for c in col if c.value is not None), default=10)
        ws.column_dimensions[col[0].column_letter].width = min(length + 2, 45)
wb.save(OUT_XLSX)

print("Excel gerado:", OUT_XLSX)


Excel gerado: Manutencoes.xlsx


In [14]:
# 5) Baixa o Excel
from google.colab import files
files.download(OUT_XLSX)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6) Salvar o Excel no Google Drive

Para salvar o arquivo `Manutencoes.xlsx` no seu Google Drive, vamos montar o Drive e depois copiar o arquivo para uma pasta de sua escolha.

In [ ]:
# Montar o Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Copiar o arquivo Excel para o Google Drive
import shutil
import os

# Defina o caminho de destino no seu Google Drive
# Por exemplo, para salvar na raiz do seu Drive, use '/content/drive/MyDrive/'
# Ou crie uma pasta específica, como '/content/drive/MyDrive/Relatorios_Manutencao/'
DRIVE_PATH = '/content/drive/MyDrive/Relatorios_Manutencao/' # <--- VOCÊ PODE MUDAR ISSO

# Crie o diretório se ele não existir
os.makedirs(DRIVE_PATH, exist_ok=True)

# Caminho completo para o arquivo de destino
destination_file_path = os.path.join(DRIVE_PATH, OUT_XLSX)

# Copie o arquivo gerado para o Google Drive
shutil.copy(OUT_XLSX, destination_file_path)

print(f"Arquivo '{OUT_XLSX}' salvo em: {destination_file_path}")

Arquivo 'Manutencoes.xlsx' salvo em: /content/drive/MyDrive/Relatorios_Manutencao/Manutencoes.xlsx
